In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import time
import os

In [16]:
def train_resnet(data_dir, num_classes, epochs=5, batch_size=8, lr=0.001, save_name='wytrenowany_resnet.pth'):

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"--- START ---")
    print(f"Device: {device} | Classes: {num_classes} | Epochs: {epochs}")

    if not os.path.exists(data_dir):
        print(f"BŁĄD: Folder {data_dir} nie istnieje!")
        return

    stat_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
    ])

    dataset = datasets.ImageFolder(root=data_dir, transform=stat_transforms)
    loader = DataLoader(dataset, batch_size=64, shuffle=False)

    mean = 0.
    std = 0.
    total_images = 0

    for images, _ in loader:
        batch_samples = images.size(0)
        images = images.view(batch_samples, images.size(1), -1)

        mean += images.mean(2).sum(0)
        std += images.std(2).sum(0)
        total_images += batch_samples

    mean /= total_images
    std /= total_images

    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean.tolist(), std.tolist())
    ])

    # 2. Dataset
    dataset = datasets.ImageFolder(root=data_dir, transform=data_transforms)
    dataset_size = len(dataset)

    # 80% train / 20% val
    train_size = int(0.8 * dataset_size)
    val_size = dataset_size - train_size

    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    print(f"Train: {train_size} | Val: {val_size}")

    # Model
    model = models.resnet18(weights=None)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    best_acc = 0.0

    # Trening
    for epoch in range(epochs):
        start_time = time.time()

        # TRAIN
        model.train()
        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels).item()

        epoch_loss = running_loss / train_size
        epoch_acc = running_corrects / train_size

        # VALIDATION
        model.eval()
        val_corrects = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                val_corrects += torch.sum(preds == labels).item()

        val_acc = val_corrects / val_size

        print(f'Epoch {epoch+1}/{epochs} | '
              f'Train Acc: {epoch_acc:.4f} | '
              f'Val Acc: {val_acc:.4f} | '
              f'Time: {time.time()-start_time:.0f}s')

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), save_name)

    print(f"Best model saved as: {save_name}")
    return model

In [13]:
model_eksperyment_1 = train_resnet(
    data_dir = './testy_nzal/FunnyBirds/train',
    num_classes = 4,
    epochs = 15,
    batch_size= 32,
    lr = 0.0001,
    save_name = 'model_hierarchia_nzal.pth'
)

--- START ---
Device: cpu | Classes: 4 | Epochs: 15
Train: 3200 | Val: 800
Epoch 1/15 | Train Acc: 0.4569 | Val Acc: 0.4925 | Time: 130s
Epoch 2/15 | Train Acc: 0.7334 | Val Acc: 0.7175 | Time: 128s
Epoch 3/15 | Train Acc: 0.9353 | Val Acc: 0.5363 | Time: 128s
Epoch 4/15 | Train Acc: 0.9731 | Val Acc: 0.7475 | Time: 129s
Epoch 5/15 | Train Acc: 0.9788 | Val Acc: 0.7688 | Time: 129s
Epoch 6/15 | Train Acc: 0.9862 | Val Acc: 0.6550 | Time: 129s
Epoch 7/15 | Train Acc: 0.9959 | Val Acc: 0.9513 | Time: 128s
Epoch 8/15 | Train Acc: 0.9969 | Val Acc: 0.8712 | Time: 128s
Epoch 9/15 | Train Acc: 0.9969 | Val Acc: 0.8788 | Time: 129s
Epoch 10/15 | Train Acc: 0.9888 | Val Acc: 0.6713 | Time: 130s
Epoch 11/15 | Train Acc: 0.9850 | Val Acc: 0.8175 | Time: 131s
Epoch 12/15 | Train Acc: 0.9953 | Val Acc: 0.8650 | Time: 129s
Epoch 13/15 | Train Acc: 0.9981 | Val Acc: 0.9300 | Time: 128s
Epoch 14/15 | Train Acc: 0.9988 | Val Acc: 0.9475 | Time: 128s
Epoch 15/15 | Train Acc: 0.9997 | Val Acc: 0.9313 | 

--- START --- lr 0.0001 batchsize 32
Device: cpu | Classes: 4 | Epochs: 15
Train: 3200 | Val: 800
Epoch 1/15 | Train Acc: 0.4569 | Val Acc: 0.4925 | Time: 130s
Epoch 2/15 | Train Acc: 0.7334 | Val Acc: 0.7175 | Time: 128s
Epoch 3/15 | Train Acc: 0.9353 | Val Acc: 0.5363 | Time: 128s
Epoch 4/15 | Train Acc: 0.9731 | Val Acc: 0.7475 | Time: 129s
Epoch 5/15 | Train Acc: 0.9788 | Val Acc: 0.7688 | Time: 129s
Epoch 6/15 | Train Acc: 0.9862 | Val Acc: 0.6550 | Time: 129s
Epoch 7/15 | Train Acc: 0.9959 | Val Acc: 0.9513 | Time: 128s
Epoch 8/15 | Train Acc: 0.9969 | Val Acc: 0.8712 | Time: 128s
Epoch 9/15 | Train Acc: 0.9969 | Val Acc: 0.8788 | Time: 129s
Epoch 10/15 | Train Acc: 0.9888 | Val Acc: 0.6713 | Time: 130s
Epoch 11/15 | Train Acc: 0.9850 | Val Acc: 0.8175 | Time: 131s
Epoch 12/15 | Train Acc: 0.9953 | Val Acc: 0.8650 | Time: 129s
Epoch 13/15 | Train Acc: 0.9981 | Val Acc: 0.9300 | Time: 128s
Epoch 14/15 | Train Acc: 0.9988 | Val Acc: 0.9475 | Time: 128s
Epoch 15/15 | Train Acc: 0.9997 | Val Acc: 0.9313 | Time: 129s
Best model saved as: model_hierarchia_nzal.pth

In [10]:
model_eksperyment_2 = train_resnet(
    data_dir = './testy_hier/FunnyBirds/train',
    num_classes = 3,
    epochs = 15,
    batch_size= 32,
    lr = 0.0001,
    save_name = 'model_hierarchia_hier.pth'
)

--- START ---
Device: cpu | Classes: 3 | Epochs: 15
Train: 2400 | Val: 600
Epoch 1/15 | Train Acc: 0.5846 | Val Acc: 0.6750 | Time: 106s
Epoch 2/15 | Train Acc: 0.7717 | Val Acc: 0.6783 | Time: 104s
Epoch 3/15 | Train Acc: 0.9033 | Val Acc: 0.5383 | Time: 102s
Epoch 4/15 | Train Acc: 0.9692 | Val Acc: 0.6783 | Time: 106s
Epoch 5/15 | Train Acc: 0.9946 | Val Acc: 0.7100 | Time: 104s
Epoch 6/15 | Train Acc: 0.9967 | Val Acc: 0.6067 | Time: 104s
Epoch 7/15 | Train Acc: 0.9971 | Val Acc: 0.8550 | Time: 99s
Epoch 8/15 | Train Acc: 0.9992 | Val Acc: 0.8633 | Time: 97s
Epoch 9/15 | Train Acc: 0.9971 | Val Acc: 0.7400 | Time: 97s
Epoch 10/15 | Train Acc: 0.9775 | Val Acc: 0.7767 | Time: 97s
Epoch 11/15 | Train Acc: 0.9788 | Val Acc: 0.8567 | Time: 97s
Epoch 12/15 | Train Acc: 0.9925 | Val Acc: 0.8350 | Time: 97s
Epoch 13/15 | Train Acc: 0.9996 | Val Acc: 0.8883 | Time: 97s
Epoch 14/15 | Train Acc: 1.0000 | Val Acc: 0.8917 | Time: 97s
Epoch 15/15 | Train Acc: 1.0000 | Val Acc: 0.9083 | Time: 97

--- START --- lr 0.0001 batchsize 32
Device: cpu | Classes: 3 | Epochs: 15
Train: 2400 | Val: 600
Epoch 1/15 | Train Acc: 0.5846 | Val Acc: 0.6750 | Time: 106s
Epoch 2/15 | Train Acc: 0.7717 | Val Acc: 0.6783 | Time: 104s
Epoch 3/15 | Train Acc: 0.9033 | Val Acc: 0.5383 | Time: 102s
Epoch 4/15 | Train Acc: 0.9692 | Val Acc: 0.6783 | Time: 106s
Epoch 5/15 | Train Acc: 0.9946 | Val Acc: 0.7100 | Time: 104s
Epoch 6/15 | Train Acc: 0.9967 | Val Acc: 0.6067 | Time: 104s
Epoch 7/15 | Train Acc: 0.9971 | Val Acc: 0.8550 | Time: 99s
Epoch 8/15 | Train Acc: 0.9992 | Val Acc: 0.8633 | Time: 97s
Epoch 9/15 | Train Acc: 0.9971 | Val Acc: 0.7400 | Time: 97s
Epoch 10/15 | Train Acc: 0.9775 | Val Acc: 0.7767 | Time: 97s
Epoch 11/15 | Train Acc: 0.9788 | Val Acc: 0.8567 | Time: 97s
Epoch 12/15 | Train Acc: 0.9925 | Val Acc: 0.8350 | Time: 97s
Epoch 13/15 | Train Acc: 0.9996 | Val Acc: 0.8883 | Time: 97s
Epoch 14/15 | Train Acc: 1.0000 | Val Acc: 0.8917 | Time: 97s
Epoch 15/15 | Train Acc: 1.0000 | Val Acc: 0.9083 | Time: 97s
Best model saved as: model_hierarchia_hier.pth

In [12]:
model_eksperyment_3 = train_resnet(
    data_dir = './testy_ABC/FunnyBirds/train',
    num_classes = 3,
    epochs = 15,
    batch_size= 32,
    lr = 0.0001,
    save_name = 'model_hierarchia_ABC.pth'
)

--- START ---
Device: cpu | Classes: 3 | Epochs: 15
Train: 24 | Val: 6
Epoch 1/15 | Train Acc: 0.3333 | Val Acc: 0.1667 | Time: 1s
Epoch 2/15 | Train Acc: 0.9167 | Val Acc: 0.1667 | Time: 1s
Epoch 3/15 | Train Acc: 0.9583 | Val Acc: 0.1667 | Time: 1s
Epoch 4/15 | Train Acc: 1.0000 | Val Acc: 0.1667 | Time: 1s
Epoch 5/15 | Train Acc: 1.0000 | Val Acc: 0.0000 | Time: 1s
Epoch 6/15 | Train Acc: 1.0000 | Val Acc: 0.1667 | Time: 1s
Epoch 7/15 | Train Acc: 1.0000 | Val Acc: 0.1667 | Time: 1s
Epoch 8/15 | Train Acc: 1.0000 | Val Acc: 0.1667 | Time: 1s
Epoch 9/15 | Train Acc: 1.0000 | Val Acc: 0.1667 | Time: 1s
Epoch 10/15 | Train Acc: 1.0000 | Val Acc: 0.1667 | Time: 1s
Epoch 11/15 | Train Acc: 1.0000 | Val Acc: 0.1667 | Time: 1s
Epoch 12/15 | Train Acc: 1.0000 | Val Acc: 0.1667 | Time: 1s
Epoch 13/15 | Train Acc: 1.0000 | Val Acc: 0.1667 | Time: 1s
Epoch 14/15 | Train Acc: 1.0000 | Val Acc: 0.1667 | Time: 1s
Epoch 15/15 | Train Acc: 1.0000 | Val Acc: 0.1667 | Time: 1s
Best model saved as: mo

In [8]:
model_eksperyment_4 = train_resnet(
    data_dir = './testy_probabilistyczne/FunnyBirds/train',
    num_classes = 2,
    epochs = 10,
    batch_size= 32,
    lr = 0.0001,
    save_name = 'model_testy_probabilistyczne.pth'
)

--- START ---
Device: cpu | Classes: 2 | Epochs: 10
Train: 1600 | Val: 400
Epoch 1/10 | Train Acc: 0.6913 | Val Acc: 0.8550 | Time: 138s
Epoch 2/10 | Train Acc: 0.9587 | Val Acc: 0.5600 | Time: 137s
Epoch 3/10 | Train Acc: 0.9950 | Val Acc: 0.9375 | Time: 137s
Epoch 4/10 | Train Acc: 0.9988 | Val Acc: 0.9300 | Time: 127s
Epoch 5/10 | Train Acc: 1.0000 | Val Acc: 0.8550 | Time: 125s
Epoch 6/10 | Train Acc: 1.0000 | Val Acc: 0.9500 | Time: 126s
Epoch 7/10 | Train Acc: 1.0000 | Val Acc: 0.9500 | Time: 125s
Epoch 8/10 | Train Acc: 0.9994 | Val Acc: 0.9525 | Time: 127s
Epoch 9/10 | Train Acc: 1.0000 | Val Acc: 0.9425 | Time: 125s
Epoch 10/10 | Train Acc: 1.0000 | Val Acc: 0.9450 | Time: 125s
Best model saved as: model_testy_probabilistyczne.pth


In [17]:
def extract_features_and_weights(data_dir, model_path, model_weights, model_features, num_classes, batch_size=64):
    """
    Ekstrahuje wektory cech (features) oraz wagi klasyfikatora (weights) 
    z wytrenowanego modelu ResNet18 i zapisuje je pod wskazanymi nazwami.
    """
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"\n--- ROZPOCZYNAM EKSTRAKCJĘ: {data_dir} ---")
    
    # 1. Obliczanie statystyk do normalizacji (identycznie jak przy treningu)
    stat_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor()
    ])
    temp_dataset = datasets.ImageFolder(root=data_dir, transform=stat_transforms)
    temp_loader = DataLoader(temp_dataset, batch_size=batch_size, shuffle=False)

    mean, std, total_images = 0., 0., 0
    for images, _ in temp_loader:
        batch_samples = images.size(0)
        images = images.view(batch_samples, images.size(1), -1)
        mean += images.mean(2).sum(0)
        std += images.std(2).sum(0)
        total_images += batch_samples

    mean /= total_images
    std /= total_images

    # 2. Przygotowanie ostatecznego Loadera
    data_transforms = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean.tolist(), std.tolist())
    ])
    dataset = datasets.ImageFolder(root=data_dir, transform=data_transforms)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False) 

    # 3. Wczytanie modelu
    model = models.resnet18(weights=None)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()

    # 4. Zapis wag przedostatniej warstwy (klasyfikatora)
    # Zapisujemy je pod nazwą podaną w 'model_weights'
    weights_to_save = model.fc.weight.detach().cpu().clone()
    torch.save(weights_to_save, model_weights)
    print(f"-> Zapisano wagi: {model_weights}")

    # 5. Ekstrakcja wektorów cech (odcinamy ostatnią warstwę)
    model.fc = nn.Identity()

    all_features = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in loader:
            inputs = inputs.to(device)
            features = model(inputs) 
            all_features.append(features.cpu())
            all_labels.append(labels.cpu())

    all_features = torch.cat(all_features)
    all_labels = torch.cat(all_labels)

    # 6. Zapis wektorów pod nazwą podaną w 'model_features'
    features_dict = {
        'features': all_features,
        'labels': all_labels
    }
    torch.save(features_dict, model_features)
    
    print(f"-> Zapisano wektory: {model_features}")
    print(f"Kształt macierzy cech: {all_features.shape}")
    print("--- ZAKOŃCZONO ---")

In [10]:
extract_features_and_weights(
    data_dir='./testy_nzal/FunnyBirds/train',
    model_path='model_hierarchia_nzal.pth', 
    model_weights='model_nzal_weights.pt',
    model_features='model_nzal_features.pt',
    num_classes=4
)


--- ROZPOCZYNAM EKSTRAKCJĘ: ./testy_nzal/FunnyBirds/train ---
-> Zapisano wagi: model_nzal_weights.pt
-> Zapisano wektory: model_nzal_features.pt
Kształt macierzy cech: torch.Size([4000, 512])
--- ZAKOŃCZONO ---


In [10]:
extract_features_and_weights(
    data_dir='./testy_hier/FunnyBirds/train',
    model_path='model_hierarchia_hier.pth', 
    model_weights='model_hier_weights.pt',
    model_features='model_hier_features.pt',
    num_classes=3
)


--- ROZPOCZYNAM EKSTRAKCJĘ: ./testy_hier/FunnyBirds/train ---
-> Zapisano wagi: model_hier_weights.pt
-> Zapisano wektory: model_hier_features.pt
Kształt macierzy cech: torch.Size([3000, 512])
--- ZAKOŃCZONO ---
